# Daily Data on Casualties (Mediazona)

In [1]:
import requests
import re
import time 
import pandas as pd
from random import uniform
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [4]:
df = pd.read_csv("../data/daily.csv")

In [5]:
def normalize_date(date_str):
    date_str = date_str.strip().lower()
    date_str = re.sub(r'^(не ранее|не позднее|до|не иранее)\s*', '', date_str)
    month_map = {
        'январь': '01', 'янв': '01',
        'февраль': '02', 'фев': '02',
        'март': '03', 'мар': '03',
        'апрель': '04', 'апр': '04',
        'май': '05', 'мая': '05',
        'июнь': '06', 'июн': '06',
        'июль': '07', 'июл': '07',
        'август': '08', 'авг': '08',
        'сентябрь': '09', 'сен': '09',
        'октябрь': '10', 'окт': '10',
        'ноябрь': '11', 'ноя': '11',
        'декабрь': '12', 'дек': '12'
    }
    
    if (re.search(r'\?|\?\?|xx|хх|весна|лето|осень|зима', date_str) and 
        not re.search(r'\d{1,2}\.\d{4}', date_str)):
        return 'NA'
    
    if re.fullmatch(r'20\d{2}', date_str):
        return 'NA'
    
    if re.fullmatch(r'20\d{2}/20\d{2}', date_str):
        return 'NA'
    
    for month_name, month_num in month_map.items():
        if month_name in date_str:
            year_match = re.search(r'20\d{2}', date_str)
            if year_match:
                return f"{month_num}.{year_match.group()}"
            return 'NA'
    
    if any(sep in date_str for sep in ['/', '-', '–']):
        date_str = re.split(r'[/–-]', date_str)[0]
    
    date_str = re.sub(r'\s+', '', date_str)
    date_str = re.sub(r'(\d)\?', r'\1', date_str)
    
    patterns = [
        r'(?P<day>\d{1,2})\.(?P<month>\d{1,2})\.(?P<year>20\d{2})',  # DD.MM.YYYY
        r'(?P<month>\d{1,2})\.(?P<year>20\d{2})',                     # MM.YYYY
        r'(?P<day>\d{1,2})\.(?P<month>\d{1,2})\.\s*(?P<year>20\d{2})' # DD.MM. YYYY
    ]
    
    for pattern in patterns:
        match = re.search(pattern, date_str)
        if match:
            month = match.group('month').zfill(2)
            year = match.group('year')
            return f"{month}.{year}"    
    return 'NA'


dates = df["death_date"]
processed_dates = [normalize_date(date) for date in dates]
df["death_month"] = processed_dates


In [ ]:
def split_names(full_name):
    try:
        surname = full_name.split(" ")[0]
        first_name = full_name.split(" ")[1]
    except IndexError:
        surname, first_name = full_name, ''
    return surname, first_name
    

surnames = []
first_names = []
for i in range(len(df)):
    surname, first_name = split_names(df.loc[i, "name"])
    surnames.append(surname)
    first_names.append(first_name)

names = pd.DataFrame(list(zip(first_names, surnames)), columns=["first_name", "last_name"])
names.to_csv("data/names.csv")

In [ ]:
df['death_month'] = pd.to_datetime(df['death_month'], format='%m.%Y', errors ='coerce')
#ethnicity = pd.read_csv("classifier/names_pred.csv")
#df = pd.concat([df, ethnicity], axis = 1)
#df["slavic"] = 0 
#df["slavic"] = (df['agr_ethnos'] == "BelRusUkr").astype(int)
result = df.dropna(subset = ['death_month']).groupby(['region', 'death_month']).agg(
    slavic_name = ('slavic', lambda x: (x == 1).sum()),
    non_slavic_name = ('slavic', lambda x: (x == 0).sum())
).reset_index() 

result

,region,death_month,slavic_name,non_slavic_name
0,Алтайский край,2022-02-01,1,0
1,Алтайский край,2022-03-01,34,2
2,Алтайский край,2022-04-01,30,2
3,Алтайский край,2022-05-01,57,5
4,Алтайский край,2022-06-01,31,3
...,...,...,...,...
2813,Ярославская область,2024-10-01,21,0
2814,Ярославская область,2024-11-01,52,2
2815,Ярославская область,2024-12-01,2,0
2816,Ярославская область,2025-02-01,8,0


In [ ]:
all_regions = df['region'].unique()
all_months = pd.date_range(
    start=df['death_month'].min(),
    end=df['death_month'].max(),
    freq='MS' 
)

complete_grid = pd.MultiIndex.from_product(
    [all_regions, all_months],
    names=['region', 'death_month']
).to_frame(index=False)


,region,death_month
0,Алтайский край,2022-02-01
1,Алтайский край,2022-03-01
2,Алтайский край,2022-04-01
3,Алтайский край,2022-05-01
4,Алтайский край,2022-06-01
...,...,...
3733,Иностранцы,2025-03-01
3734,Иностранцы,2025-04-01
3735,Иностранцы,2025-05-01
3736,Иностранцы,2025-06-01


In [9]:
final_result = (
    complete_grid.merge(
        result,
        on=['region', 'death_month'],
        how='left'
    )
    .fillna({'slavic_name': 0, 'non_slavic_name': 0})  
    .sort_values(['region', 'death_month'])
)

final_result['slavic_cumulative'] = (
    final_result.groupby('region')['slavic_name']
    .cumsum()
)

final_result['non_slavic_cumulative'] = (
    final_result.groupby('region')['non_slavic_name']
    .cumsum()
)

final_result['slavic_name %'] = final_result['slavic_cumulative']/(final_result['slavic_cumulative']+final_result["non_slavic_cumulative"])
final_result


,region,death_month,slavic_name,non_slavic_name,slavic_cumulative,non_slavic_cumulative,slavic_name %
0,Алтайский край,2022-02-01,1.0,0.0,1.0,0.0,1.000000
1,Алтайский край,2022-03-01,34.0,2.0,35.0,2.0,0.945946
2,Алтайский край,2022-04-01,30.0,2.0,65.0,4.0,0.942029
3,Алтайский край,2022-05-01,57.0,5.0,122.0,9.0,0.931298
4,Алтайский край,2022-06-01,31.0,3.0,153.0,12.0,0.927273
...,...,...,...,...,...,...,...
3649,Ярославская область,2025-03-01,4.0,1.0,492.0,16.0,0.968504
3650,Ярославская область,2025-04-01,0.0,0.0,492.0,16.0,0.968504
3651,Ярославская область,2025-05-01,0.0,0.0,492.0,16.0,0.968504
3652,Ярославская область,2025-06-01,0.0,0.0,492.0,16.0,0.968504


In [ ]:
deposits = pd.read_excel('../data/deposits.xlsx')